# Chapter 7 — A Deep Dive on Keras

Maps to Chollet Ch.7. The big idea: **progressive disclosure of complexity** — Keras gives a *spectrum*
from dead-simple `fit()` to fully hand-written training loops, and you can mix them freely.

| | easy ⟶ flexible |
|---|---|
| **build a model** | `Sequential` → **Functional API** → `Model` subclass |
| **train a model** | `fit()` → `fit()` + callbacks → **override `train_step`** → **full custom loop** |

Ch.3 covered the basics. This notebook is the *advanced, contest-useful* half: the Functional API's real
power, custom metrics/losses/callbacks, and writing the training loop yourself (your teacher's "from
scratch" wish, but inside Keras).

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np
from keras import layers, ops
import tensorflow as tf

(Xtr, ytr), (Xte, yte) = keras.datasets.mnist.load_data()
Xtr = Xtr.reshape(-1,784).astype("float32")/255; Xte = Xte.reshape(-1,784).astype("float32")/255
Xtr, ytr, Xte, yte = Xtr[:5000], ytr[:5000], Xte[:1000], yte[:1000]   # subset -> fast

def get_mnist_model():
    inputs  = keras.Input(shape=(784,))
    x       = layers.Dense(128, activation="relu")(inputs)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs)


## 1. Three ways to build — and when to use each
- **Sequential** — a list of layers. Single input → single output. Quick experiments.
- **Functional API** — a *graph* of layers. Multi-input/output, branches, layer reuse. **Use this by default.**
- **Subclassing `Model`** — write `__init__` (layers) + `call` (forward pass) in raw Python. Max flexibility
  (loops/conditionals in the forward pass), but no graph introspection (`summary`/`plot_model` show less).

Tip: layers create weights **lazily** (on first call). Declare `keras.Input(shape=...)` so `summary()`
works before training and you can watch shapes evolve.

In [ ]:
m = keras.Sequential([keras.Input(shape=(784,)),
                      layers.Dense(64, activation="relu"),
                      layers.Dense(10, activation="softmax")])
m.summary()           # works immediately because we declared Input


## 2. The Functional API's real power: multi-input / multi-output + layer reuse
Most real models are graphs, not stacks. Example: a support-ticket router with **3 inputs** (title, body,
tags) and **2 outputs** (priority = regression, department = classification). Each output gets its own loss.

In [ ]:
vocab, n_tags, n_dept = 500, 50, 4
title = keras.Input(shape=(vocab,),  name="title")
body  = keras.Input(shape=(vocab,),  name="body")
tags  = keras.Input(shape=(n_tags,), name="tags")

features  = layers.Concatenate()([title, body, tags])
features  = layers.Dense(64, activation="relu", name="mix")(features)
priority  = layers.Dense(1, activation="sigmoid", name="priority")(features)      # regression head
department= layers.Dense(n_dept, activation="softmax", name="department")(features)  # classification head

model = keras.Model([title, body, tags], [priority, department])
model.compile(optimizer="adam",
              loss=["mse", "sparse_categorical_crossentropy"],   # one loss per output
              metrics=[["mae"], ["accuracy"]])

n = 256   # dummy data just to show the fit() call shape
model.fit([np.random.randint(0,2,(n,vocab)).astype("float32"),
           np.random.randint(0,2,(n,vocab)).astype("float32"),
           np.random.randint(0,2,(n,n_tags)).astype("float32")],
          [np.random.random((n,1)), np.random.randint(0,n_dept,(n,1))],
          epochs=1, verbose=0)
print("trained a 3-input / 2-output model")


In [ ]:
# Layer reuse / feature extraction: branch a NEW output off an existing intermediate layer,
# without rebuilding or retraining the shared trunk.
shared = model.get_layer("mix").output
difficulty = layers.Dense(3, activation="softmax", name="difficulty")(shared)
model3 = keras.Model([title, body, tags], [priority, department, difficulty])
print("reused 'mix' features -> model now has", len(model3.outputs), "outputs")


## 3. Custom metric — subclass `keras.metrics.Metric`
A metric holds state in variables updated *manually* (not by backprop). Implement `update_state`,
`result`, `reset_state`. Example: root-mean-squared error over softmax outputs.

In [ ]:
class RootMeanSquaredError(keras.metrics.Metric):
    def __init__(self, name="rmse", **kw):
        super().__init__(name=name, **kw)
        self.mse_sum = self.add_weight(name="mse_sum", initializer="zeros")
        self.total   = self.add_weight(name="total",   initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = ops.one_hot(ops.cast(y_true, "int32"), ops.shape(y_pred)[1])
        self.mse_sum.assign_add(ops.sum(ops.square(y_true - y_pred)))
        self.total.assign_add(ops.cast(ops.shape(y_pred)[0], "float32"))
    def result(self):       return ops.sqrt(self.mse_sum / self.total)
    def reset_state(self):  self.mse_sum.assign(0.); self.total.assign(0.)

m = get_mnist_model()
m.compile("adam", "sparse_categorical_crossentropy",
          metrics=["accuracy", RootMeanSquaredError()])
h = m.fit(Xtr, ytr, epochs=2, batch_size=128, validation_split=0.2, verbose=0)
print("tracked custom metric ->", {k: round(v[-1],3) for k,v in h.history.items() if "rmse" in k})


## 4. Custom loss — just a function `(y_true, y_pred) → scalar`
Use `keras.ops` (backend-agnostic) so it runs on TF/JAX/PyTorch. Here: categorical cross-entropy by hand.

In [ ]:
def my_crossentropy(y_true, y_pred):
    y_true = ops.one_hot(ops.cast(y_true, "int32"), 10)
    return ops.mean(-ops.sum(y_true * ops.log(y_pred + 1e-7), axis=-1))

m = get_mnist_model()
m.compile("adam", loss=my_crossentropy, metrics=["accuracy"])
m.fit(Xtr, ytr, epochs=2, batch_size=128, verbose=0)
print("custom-loss test acc:", round(m.evaluate(Xte, yte, verbose=0, return_dict=True)["accuracy"], 3))


## 5. Callbacks — hooks that fire during `fit()`
Built-ins you'll use constantly:
- `EarlyStopping(monitor="val_loss", patience=...)` — stop when val stops improving.
- `ModelCheckpoint("best.keras", save_best_only=True)` — save the best weights.
- `ReduceLROnPlateau(factor=0.5, patience=...)` — cut the learning rate when stuck.

You can also write your own by subclassing `keras.callbacks.Callback` and overriding `on_epoch_end`, etc.

In [ ]:
class LossHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.losses = []
    def on_epoch_end(self, epoch, logs=None):
        self.losses.append(logs["loss"])
        print(f"  [custom cb] epoch {epoch}: loss={logs['loss']:.3f}")

cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    LossHistory(),
]
m = get_mnist_model(); m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(Xtr, ytr, epochs=4, batch_size=128, validation_split=0.2, callbacks=cbs, verbose=0)
print("captured losses:", [round(x,3) for x in cbs[-1].losses])


## 6. Customize `fit()` without losing it: override `train_step`
You keep all of `fit()`'s machinery (callbacks, progress bar, validation) but redefine **what happens in one
batch**. This is the modern Keras way to do custom training (GANs, custom regularizers, etc.).
*(Backend-specific; this is the TensorFlow version.)*

In [ ]:
class CustomModel(keras.Model):
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss   = self.compute_loss(y=y, y_pred=y_pred)     # uses the compiled loss
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        for metric in self.metrics:
            if metric.name == "loss": metric.update_state(loss)
            else:                     metric.update_state(y, y_pred)
        return {metric.name: metric.result() for metric in self.metrics}

inp = keras.Input(shape=(784,))
out = layers.Dense(10, activation="softmax")(layers.Dense(128, activation="relu")(inp))
cm  = CustomModel(inp, out)
cm.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
cm.fit(Xtr, ytr, epochs=2, batch_size=128, verbose=0)        # normal fit(), custom step
print("train_step override acc:", round(cm.evaluate(Xte, yte, verbose=0, return_dict=True)["accuracy"], 3))


## 7. The full custom loop (from scratch) — total control
When even `train_step` isn't enough, write the whole loop with `GradientTape`, an optimizer, metric objects,
and `tf.data` batching. This is the lowest rung — exactly what `fit()` does internally.

In [ ]:
model   = get_mnist_model()
optimizer = keras.optimizers.Adam()
loss_fn   = keras.losses.SparseCategoricalCrossentropy()
train_acc = keras.metrics.SparseCategoricalAccuracy()

train_ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(1000).batch(128)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        preds = model(x, training=True)
        loss  = loss_fn(y, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_acc.update_state(y, preds)
    return loss

for epoch in range(3):
    train_acc.reset_state()
    for xb, yb in train_ds:
        loss = train_step(xb, yb)
    print(f"epoch {epoch}: loss={float(loss):.3f}  train_acc={float(train_acc.result()):.3f}")


---
# ✍️ PROBLEMS

### P1 — Functional graph with a skip connection
Build a Functional model on MNIST where the input is added back to a hidden representation
(`layers.Add()([x, projected_input])`) — a mini residual block. Train it and compare to a plain stack.

In [ ]:
# TODO


### P2 — Custom metric: balanced accuracy / F1
Write a `keras.metrics.Metric` subclass that tracks per-class true positives and computes **macro-F1** for a
multiclass problem. Verify it against `sklearn.metrics.f1_score(average="macro")` on the test set.

In [ ]:
# TODO


### P3 — Custom training loop with validation + early stopping
Extend the §7 loop to also run a **validation** pass each epoch (separate metric), print train/val accuracy,
and **stop early** when val accuracy hasn't improved for 3 epochs (keep the best weights with
`model.get_weights()` / `set_weights()`).

In [ ]:
# TODO


### P4 — Override train_step for label smoothing
Subclass `keras.Model` and override `train_step` so the loss uses **label smoothing** (mix the one-hot target
with a uniform distribution, e.g. 0.9 true + 0.1/K). Compare test accuracy with and without smoothing.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Functional API (default choice; multi-IO)

In [ ]:
import keras
from keras import layers
inp_a = keras.Input(shape=(DA,), name="a")
inp_b = keras.Input(shape=(DB,), name="b")
x = layers.Concatenate()([inp_a, inp_b])
x = layers.Dense(64, activation="relu")(x)
out1 = layers.Dense(1, activation="sigmoid", name="reg")(x)
out2 = layers.Dense(C, activation="softmax", name="cls")(x)
model = keras.Model([inp_a, inp_b], [out1, out2])
model.compile("adam", loss=["mse", "sparse_categorical_crossentropy"],
              metrics=[["mae"], ["accuracy"]])


### T2 — Custom metric

In [ ]:
from keras import ops
import keras
class MyMetric(keras.metrics.Metric):
    def __init__(self, name="my_metric", **kw):
        super().__init__(name=name, **kw)
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        ...  # accumulate into self.total / self.count
    def result(self):      return self.total / self.count
    def reset_state(self): self.total.assign(0.); self.count.assign(0.)


### T3 — Custom loss (backend-agnostic via keras.ops)

In [ ]:
from keras import ops
def my_loss(y_true, y_pred):
    return ops.mean(ops.square(y_true - y_pred))   # example: MSE
# model.compile(optimizer="adam", loss=my_loss, metrics=[...])


### T4 — Override train_step (keep fit(), change the math) — TF backend

In [ ]:
import tensorflow as tf, keras
class CustomModel(keras.Model):
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compute_loss(y=y, y_pred=y_pred)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        for m in self.metrics:
            m.update_state(loss) if m.name == "loss" else m.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}


### T5 — Full custom training loop

In [ ]:
import tensorflow as tf, keras
opt = keras.optimizers.Adam(); loss_fn = keras.losses.SparseCategoricalCrossentropy()
acc = keras.metrics.SparseCategoricalAccuracy()
ds  = tf.data.Dataset.from_tensor_slices((X, y)).shuffle(1000).batch(128)
@tf.function
def step(x, y):
    with tf.GradientTape() as tape:
        p = model(x, training=True); loss = loss_fn(y, p)
    g = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(g, model.trainable_variables)); acc.update_state(y, p); return loss
for epoch in range(EPOCHS):
    acc.reset_state()
    for xb, yb in ds: loss = step(xb, yb)
    print(epoch, float(loss), float(acc.result()))


---
### ✅ Checklist
- [ ] Pick Sequential / Functional / subclass appropriately; explain lazy weight creation + `Input`.
- [ ] Build a multi-input/multi-output Functional model with one loss per output.
- [ ] Reuse an intermediate layer's output to branch a new head (feature extraction).
- [ ] Write a custom metric (update_state/result/reset_state) and a custom loss.
- [ ] Use EarlyStopping/ModelCheckpoint/ReduceLROnPlateau + a custom callback.
- [ ] Override `train_step`, and also write a full `GradientTape` training loop from scratch.

**Next: Chapter 8** — *Image classification*: convolutions, pooling, and your first ConvNet (plus the
from-scratch CNN your lab cares about). Say "Chapter 8".